# Vast.ai training

Upload this notebook to `/workspace` on your single-GPU **NVIDIA CUDA** instance. Set your Hugging Face write token below (or supply `HF_TOKEN` in the instance environment), then **Run All**.

Every attempt uses a new `/workspace/cm-study` folder: clone the source, install Python/dependencies, download the dataset, extract `vizwiz/`, and delete the ZIP. All selected models train from scratch, one at a time, for 30 epochs. Only `best.pt`, training history/settings, validation/test metrics and one `logits.npz` per split, and study summaries are saved. Set `SAVE_MASKS=True` for optional PNGs.

Completed models upload to a new timestamped folder in your public Hugging Face repository. There is no continuation, restoration, or reuse of earlier results. If anything fails, **delete `/workspace/cm-study` and Run All from the beginning**. The notebook does not delete it for you. Interrupting the run cell stops its processes.


In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Or paste your write token here; keep the notebook private.
SAVE_MASKS = False  # Optional PNG export; logits are always saved.
RUN_IDS = []  # All 13 runs. Example: ["cm_shared_s42", "cm_shared_s123", "cm_shared_s456"]

WORK_DIR = "/workspace/cm-study"
GIT_URL = "https://github.com/DPTPhatUS/VizWiz-VQA-Grounding.git"
GIT_BRANCH = "exp/context-membership"
HF_REPO = "Last-Bullet/vizwiz-cm-aciids-2027"
HF_PREFIX = "matched-30ep-v1"  # Choose a new prefix for a separate study.


In [ ]:
from pathlib import Path
import signal
import subprocess
import sys

work = Path(WORK_DIR)
work.mkdir(parents=True)
repo = work / "source"
env = os.environ.copy()
env.update(HF_HOME=str(work / "cache/huggingface"), UV_CACHE_DIR=str(work / "cache/uv"),
           UV_PYTHON_INSTALL_DIR=str(work / "python"), UV_PYTHON_PREFERENCE="only-managed",
           TORCH_HOME=str(work / "cache/torch"), MPLCONFIGDIR=str(work / "cache/matplotlib"),
           MPLBACKEND="Agg", OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false",
           PYTHONUNBUFFERED="1")
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

def run(command, cwd=None):
    process = subprocess.Popen([str(part) for part in command], cwd=cwd, env=env, start_new_session=True)
    try:
        code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
        raise
    if code:
        raise subprocess.CalledProcessError(code, command)

run(["git", "clone", "--single-branch", "--branch", GIT_BRANCH, GIT_URL, repo])
run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--force-reinstall", "uv"])
run([sys.executable, "-m", "uv", "python", "install", "3.12"])
run([sys.executable, "-m", "uv", "sync", "--frozen", "--python", "3.12"], cwd=repo)

command = [repo / ".venv/bin/python", "-u", "-m", "experiments.vast", "--work-dir", work,
           "--hf-repo", HF_REPO, "--hf-prefix", HF_PREFIX]
if SAVE_MASKS:
    command += ["--save-masks"]
if RUN_IDS:
    command += ["--runs", *RUN_IDS]
run(command, cwd=repo)
